In [ ]:
import DataLoaderPipeline as dplmodule
from DataLoaderPipeline import DataLoaderPipeline
import importlib
importlib.reload(dplmodule)
DATA_PATH = "data/CASIA_FASD_V3_224x224/DATA"
DATA_MAP_PATH = "data_maps/CASIA_FASD_V3_30percent_uniformSampling.json"

all_subjects = [f"{i:02d}" for i in range(1, 51)]  # Generate subject IDs from '01' to '50'.
train_subs = all_subjects[:20]
val_subs = all_subjects[20:25] 
test_subs = all_subjects[20:] 
config={
        "data_params": {
            "dataset_path": "data/CASIA_FASD_v3_224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "data_maps/CASIA_FASD_v3_all.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 0,
            "brightness_range": [0.8,1.2],
            "zoom_range": 0.1
        },
      
    }

dlp = DataLoaderPipeline(data_params=config["data_params"],
                         filtering_params=config["filtering_params"],
                         augmentation_params=config["augmentation_params"])


train_ds = dlp.build_pipeline(train_subs, balanced=True, augment=True)
val_ds   = dlp.build_pipeline(val_subs, balanced=False, augment=False)
test_ds  = dlp.build_pipeline(test_subs, balanced=False, augment=False)
for ds in [train_ds, val_ds, test_ds]:
    dlp.audit_dataset(ds, batchs=20)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_multi_colorspace_model(input_shape=(224, 224, 3)):
    # 1. Input Layer (RGB)
    input_rgb = layers.Input(shape=input_shape, name="input_rgb")
    
    # 2. Color Space Transformations
    # Note: TF expects images in [0, 1] for these built-in functions
    input_hsv = layers.Lambda(lambda x: tf.image.rgb_to_hsv(x), name="rgb_to_hsv")(input_rgb)
    input_ycbcr = layers.Lambda(lambda x: tf.image.rgb_to_yuv(x), name="rgb_to_ycbcr")(input_rgb)
    
    # 3. Concatenate all 9 channels (3 RGB + 3 HSV + 3 YCbCr)
    # Shape transition: (Batch, 224, 224, 3) -> (Batch, 224, 224, 9)
    merged_channels = layers.Concatenate(axis=-1, name="nine_channel_fusion")(
        [input_rgb, input_hsv]
    )
    
    # 4. Feature Extraction (Example: Lightweight CNN for Spoofing)
    # Since you have 9 channels, the first Conv layer will learn cross-color correlations
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(merged_channels)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    
    # 5. Output Layer (Binary Classification: Real vs Spoof)
    output = layers.Dense(1, activation='sigmoid', name="classifier")(x)
    
    model = models.Model(inputs=input_rgb, outputs=output, name="MultiColorspace_Spoofing_Detector")
    return model

# Instantiate for your TrainingPipeline
model = build_multi_colorspace_model()
model.summary()

In [ ]:

def build_mc_cdcn_model(input_shape=(224, 224, 3), theta=0.7):
    """
    Builds the MC-CDCN model for Intelligent Facial Spoofing Detection.
    Combines RGB, HSV, YCrCb with Central Difference Convolutions.
    """
    
    # --- Custom CDC Layer inside the function for portability ---
    class CDC2D(layers.Layer):
        def __init__(self, filters, kernel_size=3, theta=0.7, **kwargs):
            super().__init__(**kwargs)
            self.filters, self.kernel_size, self.theta = filters, kernel_size, theta
            self.conv = layers.Conv2D(filters, kernel_size, padding='same', use_bias=False)
        def call(self, x):
            out_normal = self.conv(x)
            kernel_sum = tf.reduce_sum(self.conv.kernel, axis=(0, 1))
            out_center = tf.nn.convolution(x, tf.reshape(kernel_sum, (1, 1, x.shape[-1], self.filters)), padding='SAME')
            return out_normal - self.theta * out_center
        def get_config(self):
            return {**super().get_config(), "filters": self.filters, "kernel_size": self.kernel_size, "theta": self.theta}

    # --- 1. Inputs & Multi-Chromatic Fusion ---
    input_rgb = layers.Input(shape=input_shape, name="input_rgb")
    
    # Normalize and transform color spaces
    # Note: Assumes input is [0, 255], scales to [0, 1] for transformations
    x_norm = layers.Rescaling(1./255)(input_rgb)
    input_hsv = layers.Lambda(lambda x: tf.image.rgb_to_hsv(x), name="hsv_transform")(x_norm)
    input_ycbcr = layers.Lambda(lambda x: tf.image.rgb_to_yuv(x), name="ycbcr_transform")(x_norm)
    
    merged = layers.Concatenate(axis=-1, name="9_channel_fusion")([x_norm, input_hsv, input_ycbcr])

    # --- 2. Feature Extraction Blocks (CDC + Residual) ---
    def cdc_res_block(x, filters, stride=1):
        shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same')(x)
        
        # Branch with Central Difference
        val = CDC2D(filters, theta=theta)(x)
        val = layers.BatchNormalization()(val)
        val = layers.Activation('relu')(val)
        if stride > 1: val = layers.MaxPooling2D((stride, stride))(val)
        
        val = layers.Conv2D(filters, (3, 3), padding='same')(val)
        val = layers.BatchNormalization()(val)
        
        return layers.Add()([shortcut, val])

    # Block 1: Initial texture extraction
    x = cdc_res_block(merged, 64)
    x = layers.MaxPooling2D((2, 2))(x) # 112x112

    # Block 2: Middle-level features with Spatial Attention
    x = cdc_res_block(x, 128)
    attn = layers.Conv2D(1, (1, 1), activation='sigmoid')(x)
    x = layers.Multiply()([x, attn])
    x = layers.MaxPooling2D((2, 2))(x) # 56x56

    # Block 3: High-level semantics
    x = cdc_res_block(x, 256)

    # --- 3. Global Head ---
    # Combine Average and Max pooling to catch both global structure and local "glints"
    gap = layers.GlobalAveragePooling2D()(x)
    gmp = layers.GlobalMaxPooling2D()(x)
    combined = layers.Concatenate()([gap, gmp])
    
    x = layers.Dense(128, activation='relu', kernel_regularizer='l2')(combined)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid', name="classifier")(x)

    return models.Model(inputs=input_rgb, outputs=output, name="MC_CDCN")
build_mc_cdcn_model().summary()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the layers you want to visualize
layer_outputs = [
    model.get_layer("nine_channel_fusion").output,
    model.get_layer("conv2d_1").output
]

# 2. Create a model that returns these outputs
visualization_model = tf.keras.models.Model(inputs=model.input, outputs=layer_outputs)

# 3. Get a sample from your validation dataset (CASIA-FASD)
# Assuming val_ds is a batched tf.data.Dataset
sample_batch, _ = next(iter(val_ds))
single_img = sample_batch[2:3] # Shape: (1, 224, 224, 3)

# 4. Predict to get the feature maps
fusion_output, conv_output = visualization_model.predict(single_img)

In [ ]:
def plot_nine_channels(fusion_data):
    channel_names = ['R', 'G', 'B', 'H', 'S', 'V', 'Y', 'Cb', 'Cr']
    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(fusion_data[0, :, :, i], cmap='viridis')
        plt.title(f"Channel: {channel_names[i]}")
        plt.axis('off')
    plt.show()

plot_nine_channels(fusion_output)

In [ ]:
def plot_conv_layers(conv_data, num_filters=16):
    plt.figure(figsize=(16, 8))
    for i in range(num_filters):
        plt.subplot(2, 8, i + 1)
        plt.imshow(conv_data[0, :, :, i], cmap='magma')
        plt.axis('off')
    plt.suptitle("First Convolutional Layer Learned Features")
    plt.show()

plot_conv_layers(conv_output)

In [ ]:
plt.imshow(single_img[0])


In [ ]:
# 1. Input Layer (RGB)
input_rgb = layers.Input(shape=(244, 244, 3), name="input_rgb")

# 2. Color Space Transformations
# Note: TF expects images in [0, 1] for these built-in functions
input_hsv = layers.Lambda(lambda x: tf.image.rgb_to_hsv(x), name="rgb_to_hsv")(input_rgb)
input_ycbcr = layers.Lambda(lambda x: tf.image.rgb_to_yuv(x), name="rgb_to_ycbcr")(input_rgb)

In [ ]:
# Get your input data
sample_input = tf.random.uniform((1, 224, 224, 3))

# Access the layer by name and call it like a function
layer = model.get_layer("rgb_to_hsv")
output_tensor = layer(single_img) 

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
channel_names = ['Y (Luminance)', 'Cb (Blue-diff)', 'Cr (Red-diff)']
# 1. Get the output (Pass your single image)
output_tensor = layer(single_img) 

# 2. Remove the batch dimension (1, 224, 224, 3) -> (224, 224, 3)
# and convert to numpy
img_to_plot = output_tensor[0].numpy()

for i in range(3):
    axes[i].imshow(img_to_plot[:, :, i], cmap='gray')
    axes[i].set_title(channel_names[i])
    axes[i].axis('off')

plt.show()

# configs

In [ ]:

configs_example = [
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_10percent_BMSelection.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 20,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_30percent_BMSelection.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 20,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_50percent_uniformSampling.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 20,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_all.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 20,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1500,
            "initial_epochs": 50
        }      
    },
]


In [ ]:

configs_example = [
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_10percent_BMSelection.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 10,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_30percent_BMSelection.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 10,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_30percent_uniformSampling.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 10,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_50percent_uniformSampling.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 10,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1000,
            "initial_epochs": 50
        }      
    },
    {
        "data_params": {
            "dataset_path": "/kaggle/input/datasets/maameri/visual-computing-thesis-data-224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32                
        },
        
        "filtering_params": {
            "data_map_path": "/kaggle/working/face-anti-spoofing-pipline/data_maps/CASIA_FASD_V3_all.json",
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 1,
            "brightness_range": [0.9,1.1],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None  # To be filled after model initialization
                       
        },
        
        "training_params": {
            "early_stopping_patience": 10,
            "learning_rate": 0.00005,
            "steps_per_epoch": 1500,
            "initial_epochs": 50
        }      
    },
]
